In [3]:
import os

_ffmpeg = r"C:\Users\Hi\AppData\Local\ffmpegio\ffmpeg-downloader\ffmpeg\bin"
os.environ["PATH"] += os.pathsep + _ffmpeg

print("FFmpeg path loaded successfully!")

FFmpeg path loaded successfully!


In [ ]:
import whisper
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1: ADJUST FILE & LANGUAGE
audio_path = "sample_video_vie.mp4"

# vi / en
lang = "vi"

print("Downloading AI models...")
asr_model = whisper.load_model("large-v3-turbo")
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2: ASR and STORE METADATA
metadata_list = []
text_segments = []

print(f"\nProcessing file: {audio_path} (Language: {lang})")

# LANGUAGE
result = asr_model.transcribe(audio_path, language=lang)

for segment in result["segments"]:
    segment_text = segment["text"].strip()
    text_segments.append(segment_text)
    
    metadata_list.append({
        "audio_file": audio_path,
        "start_time": segment["start"],
        "end_time": segment["end"],
        "text_segment": segment_text
    })

print(f"Successfully extracted {len(metadata_list)} segments from this file.")

# 3: FAISS and SEARCH
print("\nCreating FAISS Index...")
segment_embeddings = text_model.encode(text_segments)
faiss.normalize_L2(segment_embeddings)

dimension = segment_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(segment_embeddings)

# QUERY
query_text = "giới phân tích dự báo đạt 6.000 tỷ đô trong 3 năm tới" 

print(f"\n---> TOP 10 RESULTS FOR: '{query_text}' <---")
query_embedding = text_model.encode([query_text])
faiss.normalize_L2(query_embedding)

k = min(10, len(metadata_list)) 
scores, indices = index.search(query_embedding, k)

for i in range(k):
    match_idx = indices[0][i]
    meta = metadata_list[match_idx]
    
    print(f"Top {i+1} | Score: {scores[0][i]:.3f} | File: {meta['audio_file']}")
    print(f"Time: {meta['start_time']:.2f}s -> {meta['end_time']:.2f}s")
    print(f"Content: {meta['text_segment']}")
    print("-" * 50)

100%|█████████████████████████████████████| 1.51G/1.51G [00:34<00:00, 47.5MiB/s]
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6837.06it/s]



Processing file: sample_video_vie.mp4 (Language: vi)


c:\Users\Hi\AppData\Local\Programs\Python\Python314\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Successfully extracted 7 segments from this file.

Creating FAISS Index...

---> TOP 10 RESULTS FOR: 'What if you could hear what your car is thinking' <---
Top 1 | Score: 0.086 | File: sample_video_vie.mp4
Time: 49.28s -> 55.50s
Content: CEO Jason Huang được mệnh danh là biểu tượng của làn sóng AI với vai trò lớn trong nhiều sáng kiến chiến lược cấp quốc gia.
--------------------------------------------------
Top 2 | Score: 0.063 | File: sample_video_vie.mp4
Time: 20.30s -> 29.66s
Content: Việc Nvidia vượt ngưỡng vốn hóa 4.000 tỷ USD không chỉ phản ánh sự bùng nổ về tài chính mà còn cho thấy vai trò then chốt của công ty này trong hệ sinh thái trí tuệ nhân tạo toàn cầu.
--------------------------------------------------
Top 3 | Score: 0.061 | File: sample_video_vie.mp4
Time: 30.00s -> 42.02s
Content: Từ một hãng sản xuất chip đồ họa phục vụ game thủ, Nvidia đã chuyển mình thành trung tâm quyền lực mới trong hoàn công nghệ với các sản phẩm hỗ trợ đào tạo mô hình AI, vận hành robot, xe 